In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = Path("../data")

SENSORS = {
    'PS1': 6000, 'PS2': 6000, 'PS3': 6000, 'PS4': 6000, 'PS5': 6000, 'PS6': 6000,
    'EPS1': 6000,
    'FS1': 600,  'FS2': 600,
    'TS1': 60,   'TS2': 60,   'TS3': 60,   'TS4': 60,
    'VS1': 60,   
    'CE': 60,    
    'CP': 60,    
    'SE': 60
}

def extract_features(sensor_array):
    return {
        'mean': np.mean(sensor_array),
        'std': np.std(sensor_array),
        'min': np.min(sensor_array),
        'max': np.max(sensor_array),
        'range': np.max(sensor_array) - np.min(sensor_array),
        'rms': np.sqrt(np.mean(sensor_array**2)),
        'skewness': stats.skew(sensor_array),
        'kurtosis': stats.kurtosis(sensor_array)
    }

# Cargar sensores
print("Loading sensor data...")
sensor_data = {}
for sensor in SENSORS:
    sensor_data[sensor] = pd.read_csv(
        DATA_PATH / f"{sensor}.txt", sep='\t', header=None
    )

# Cargar targets
targets = pd.read_csv(
    DATA_PATH / "profile.txt",
    sep='\t', header=None,
    names=['cooler', 'valve', 'pump', 'accumulator', 'stable']
)

# Filtrar ciclos estables
stable_mask = targets['stable'] == 0
targets = targets[stable_mask].reset_index(drop=True)

# Feature engineering
print("Extracting features...")
all_features = []
stable_indices = targets.index.tolist()

for cycle_idx in range(len(sensor_data['PS1'])):
    if not stable_mask.iloc[cycle_idx]:
        continue
    cycle_features = {}
    for sensor_name, sensor_df in sensor_data.items():
        measurements = sensor_df.iloc[cycle_idx].values
        features = extract_features(measurements)
        for feat_name, feat_value in features.items():
            cycle_features[f"{sensor_name}_{feat_name}"] = feat_value
    all_features.append(cycle_features)

X = pd.DataFrame(all_features)

print(f"X shape: {X.shape}")
print(f"Targets shape: {targets.shape}")
print("Ready for training.")

Loading sensor data...
Extracting features...
X shape: (1449, 136)
Targets shape: (1449, 5)
Ready for training.


In [5]:
# Definir los cuatro targets con sus labels descriptivos
TARGETS = {
    'cooler': {3: 'near_failure', 20: 'reduced_efficiency', 100: 'full_efficiency'},
    'valve': {73: 'near_failure', 80: 'severe_lag', 90: 'small_lag', 100: 'optimal'},
    'pump': {0: 'no_leakage', 1: 'weak_leakage', 2: 'severe_leakage'},
    'accumulator': {90: 'near_failure', 100: 'severely_reduced', 
                    115: 'slightly_reduced', 130: 'optimal'}
}

# Mapear valores numéricos a labels descriptivos
y = {}
for component, label_map in TARGETS.items():
    y[component] = targets[component].map(label_map)
    print(f"{component}: {y[component].unique()}")

# Split estratificado — mantiene proporción de clases en train y test
# Usamos cooler como referencia para la estratificación
X_train, X_test, y_train, y_test = {}, {}, {}, {}

# Primero dividimos X con el target de cooler como referencia
X_tr, X_te, _, _ = train_test_split(
    X, y['cooler'],
    test_size=0.2,
    random_state=42,
    stratify=y['cooler']
)

train_idx = X_tr.index
test_idx = X_te.index

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]

for component in TARGETS:
    y_train[component] = y[component].loc[train_idx]
    y_test[component] = y[component].loc[test_idx]

print(f"\nTrain size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

# Configurar MLflow
mlflow.set_experiment("hydraulic_system_monitoring")
print("\nMLflow experiment configured.")

cooler: ['near_failure' 'reduced_efficiency' 'full_efficiency']
valve: ['near_failure' 'severe_lag' 'small_lag' 'optimal']
pump: ['severe_leakage' 'weak_leakage' 'no_leakage']
accumulator: ['optimal' 'slightly_reduced' 'severely_reduced' 'near_failure']

Train size: 1159
Test size: 290


2026/04/23 16:15:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/23 16:15:40 INFO mlflow.store.db.utils: Updating database tables
2026/04/23 16:15:42 INFO mlflow.tracking.fluent: Experiment with name 'hydraulic_system_monitoring' does not exist. Creating a new experiment.



MLflow experiment configured.


In [6]:
def train_and_log(model, model_name, component, X_train, y_train, X_test, y_test):
    """
    Entrena un modelo, evalúa y loggea todo en MLflow.
    
    Params:
        model      → instancia del modelo sklearn (sin entrenar)
        model_name → nombre descriptivo para el run
        component  → nombre del componente hidráulico
        X_train, y_train → datos de entrenamiento
        X_test, y_test   → datos de evaluación
    
    Returns:
        dict con métricas del modelo
    """
    with mlflow.start_run(run_name=f"{model_name}_{component}"):
        # Pipeline: scaler + modelo
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        # StandardScaler normaliza las features a media 0 y std 1
        # Importante porque los sensores tienen escalas muy distintas:
        # PS1 mide en bar (~150), TS1 en °C (~40), VS1 en mm/s (~1)
        # Sin normalizar los modelos que usan distancias se ven afectados

        # Cross-validation en train — estimación robusta del error
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(
            pipeline, X_train, y_train,
            cv=cv, scoring='f1_weighted', n_jobs=-1
        )
        # f1_weighted → promedia F1 por clase ponderando por cantidad de muestras
        # n_jobs=-1   → usa todos los cores disponibles para paralelizar

        # Entrenar en todo el train set
        pipeline.fit(X_train, y_train)

        # Evaluar en test
        y_pred = pipeline.predict(X_test)
        test_f1 = f1_score(y_test, y_pred, average='weighted')
        test_acc = accuracy_score(y_test, y_pred)

        # Loggear en MLflow
        mlflow.log_param("model", model_name)
        mlflow.log_param("component", component)
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_metric("cv_f1_mean", cv_scores.mean())
        mlflow.log_metric("cv_f1_std", cv_scores.std())
        mlflow.log_metric("test_f1", test_f1)
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.sklearn.log_model(pipeline, f"model_{component}")

        print(f"\n{model_name} — {component.upper()}")
        print(f"  CV F1:   {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
        print(f"  Test F1: {test_f1:.3f}")
        print(f"  Test Acc: {test_acc:.3f}")
        print(classification_report(y_test, y_pred))

        return {
            'model_name': model_name,
            'component': component,
            'cv_f1_mean': cv_scores.mean(),
            'cv_f1_std': cv_scores.std(),
            'test_f1': test_f1,
            'test_accuracy': test_acc,
            'pipeline': pipeline
        }

In [7]:
# Imputación de NaN
print(f"NaN antes: {X.isnull().sum().sum()}")

X_clean = X.fillna(0)

X_train = X_clean.loc[train_idx]
X_test = X_clean.loc[test_idx]

print(f"NaN después: {X_clean.isnull().sum().sum()}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

NaN antes: 1544
NaN después: 0
Train: (1159, 136), Test: (290, 136)


In [8]:
# Definir modelos a comparar
models = {
    'LogisticRegression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )
}
# class_weight='balanced' → ajusta los pesos de las clases automáticamente
# inversamente proporcional a su frecuencia en el dataset
# Importante para valve y accumulator que tienen clases desbalanceadas

# Entrenar todos los modelos en todos los componentes
results = []

for component in TARGETS:
    print(f"\n{'='*50}")
    print(f"Training models for: {component.upper()}")
    print('='*50)
    
    for model_name, model in models.items():
        # XGBoost necesita LabelEncoder — no acepta strings como target
        if model_name == 'XGBoost':
            le = LabelEncoder()
            y_train_encoded = le.fit_transform(y_train[component])
            y_test_encoded = le.transform(y_test[component])
            
            result = train_and_log(
                xgb.XGBClassifier(
                    n_estimators=100,
                    random_state=42,
                    eval_metric='mlogloss',
                    verbosity=0
                ),
                model_name, component,
                X_train, y_train_encoded,
                X_test, y_test_encoded
            )
        else:
            result = train_and_log(
                model, model_name, component,
                X_train, y_train[component],
                X_test, y_test[component]
            )
        
        results.append(result)

print("\n✓ Training complete.")


Training models for: COOLER


2026/04/23 16:15:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 16:15:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



LogisticRegression — COOLER
  CV F1:   1.000 ± 0.000
  Test F1: 1.000
  Test Acc: 1.000
                    precision    recall  f1-score   support

   full_efficiency       1.00      1.00      1.00        98
      near_failure       1.00      1.00      1.00        96
reduced_efficiency       1.00      1.00      1.00        96

          accuracy                           1.00       290
         macro avg       1.00      1.00      1.00       290
      weighted avg       1.00      1.00      1.00       290



2026/04/23 16:16:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 16:16:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



RandomForest — COOLER
  CV F1:   1.000 ± 0.000
  Test F1: 1.000
  Test Acc: 1.000
                    precision    recall  f1-score   support

   full_efficiency       1.00      1.00      1.00        98
      near_failure       1.00      1.00      1.00        96
reduced_efficiency       1.00      1.00      1.00        96

          accuracy                           1.00       290
         macro avg       1.00      1.00      1.00       290
      weighted avg       1.00      1.00      1.00       290



...


LogisticRegression — ACCUMULATOR
  CV F1:   0.873 ± 0.018
  Test F1: 0.900
  Test Acc: 0.900
                  precision    recall  f1-score   support

    near_failure       0.92      0.99      0.95        72
         optimal       0.93      0.89      0.91        70
severely_reduced       0.90      0.86      0.88        85
slightly_reduced       0.85      0.87      0.86        63

        accuracy                           0.90       290
       macro avg       0.90      0.90      0.90       290
    weighted avg       0.90      0.90      0.90       290



2026/04/23 16:17:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 16:17:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



RandomForest — ACCUMULATOR
  CV F1:   0.985 ± 0.007
  Test F1: 0.993
  Test Acc: 0.993
                  precision    recall  f1-score   support

    near_failure       1.00      0.99      0.99        72
         optimal       1.00      1.00      1.00        70
severely_reduced       0.99      0.99      0.99        85
slightly_reduced       0.98      1.00      0.99        63

        accuracy                           0.99       290
       macro avg       0.99      0.99      0.99       290
    weighted avg       0.99      0.99      0.99       290



2026/04/23 16:17:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 16:17:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



XGBoost — ACCUMULATOR
  CV F1:   0.978 ± 0.012
  Test F1: 0.990
  Test Acc: 0.990
              precision    recall  f1-score   support

           0       0.99      0.99      0.99        72
           1       0.99      1.00      0.99        70
           2       1.00      0.99      0.99        85
           3       0.98      0.98      0.98        63

    accuracy                           0.99       290
   macro avg       0.99      0.99      0.99       290
weighted avg       0.99      0.99      0.99       290


✓ Training complete.


In [9]:
import joblib
from pathlib import Path

MODELS_PATH = Path("../models")
MODELS_PATH.mkdir(exist_ok=True)

# Mejores modelos por componente (según test F1)
best_models = {
    'cooler': ('RandomForest', results[1]['pipeline']),      # idx 1
    'valve': ('RandomForest', results[4]['pipeline']),       # idx 4
    'pump': ('XGBoost', results[8]['pipeline']),             # idx 8
    'accumulator': ('RandomForest', results[10]['pipeline']) # idx 10
}

for component, (model_name, pipeline) in best_models.items():
    path = MODELS_PATH / f"{component}_model.pkl"
    joblib.dump(pipeline, path)
    print(f"✓ {component} ({model_name}) → {path}")

✓ cooler (RandomForest) → ../models/cooler_model.pkl
✓ valve (RandomForest) → ../models/valve_model.pkl
✓ pump (XGBoost) → ../models/pump_model.pkl
✓ accumulator (RandomForest) → ../models/accumulator_model.pkl
